# Llama3.2-3B / ZU7CG / PYNQ 4.0 / 187.5 MHz

部署目录：`llama32_3b_ctx8192_250mhz`。本目录已有硬件文件，可先进行板端功能验证。

在板卡 Jupyter 中选择安装了 PYNQ 4.0 的 Python 内核，按顺序逐格执行。需要本目录已有 `hw/` 三件套、两份模型镜像和 tokenizer；本 Notebook 不修改启动设备树。每步成功后再继续，保留完整报错。Overlay 下载、寄存器及物理 DDR 访问需要该内核有相应系统权限。

模型只加载一次，最后的文本生成单元可以重复执行。结果保存在项目的 `logs/`。

## 1. 确认内核、文件和启动内存预留

这一格不下载 bit、不写 DDR。PS 模型池需要在当前运行设备树中将 `[0x800000000, 0x880000000)` 专用预留为 2 GiB `no-map`；如果失败，先处理启动设备树。

如果提示缺少依赖，可在新的单元中运行 `%pip install -r ../requirements.txt`（Notebook 工作目录为 `notebooks/` 时），完成后重启内核，再从本格开始。

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import sys
import pynq

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from llama32_pynq import BoardConfig
from llama32_pynq.board import PynqSession, inspect_files
from llama32_pynq.memory import assert_ps_reserved

print("Python:", sys.executable)
print("PYNQ:", pynq.__version__, "UID:", os.geteuid())
print("Project:", ROOT)
assert str(pynq.__version__).split('.')[:2] == ['4', '0'], "请选择 PYNQ 4.0 内核"
config = BoardConfig.load(ROOT / "config/pynq.json")
report = {"project": "llama32_3b_ctx8192_250mhz", "pynq_version": str(pynq.__version__),
          "python": sys.executable, "started_at": datetime.now().astimezone().isoformat()}
report["files"] = inspect_files(config)
print(json.dumps(report["files"], ensure_ascii=False, indent=2))
assert report["files"]["inference_assets_ready"], "部署资源检查失败，请查看上面的 inference_blockers"
report["reservation"] = assert_ps_reserved()
print(report["reservation"])

## 2. 下载 Overlay 并检查控制寄存器

本格配置 PL 时钟并下载本工程 bit/HWH。检查输出中的 magic 为 77（十六进制 0x4d），scratch 读回测试通过。

In [ ]:
session = PynqSession(config)
report["connect"] = session.connect()
report["connect"]

## 3. 双 DDR 小范围读回

分别读写并恢复两片专用 DDR 末页的 64 字节。输出的 PS、PL 均应有 `restored=True`。这是 CPU 通路的读回检查，不代替后续模型加载和加速器推理测试。

In [ ]:
report["ddr"] = session.ddr_smoke_test()
report["ddr"]

## 4. 加载模型并完整读回校验

两份模型各 1,427,628,032 字节，本格会检查 manifest SHA、写入 PS/PL DDR 并完整读回。等待本格执行完成。两份 `readback_verified` 均应为 True。

In [ ]:
manifest = json.loads((ROOT / "data/model_manifest.json").read_text())
hashes = dict(zip(("ps", "pl"), (c["image"]["sha256"] for c in manifest["cores"])))
last_progress = {}
def progress(bank, done, total):
    if done == total or done - last_progress.get(bank, 0) >= 256 * 1024 * 1024:
        print(f"{bank.upper()} DDR: {done / 2**20:.0f}/{total / 2**20:.0f} MiB", flush=True)
        last_progress[bank] = done

report["load"] = session.load_models(progress=progress, expected_hashes=hashes)
report["load"]

## 5. BOS 首 token 和重复测试

BOS 为 128000，输出最多两个 token。两次结果相同只证明该测试的重复性；数值正确性仍需与相同量化镜像的参考对照。若遇 EOS 提前结束，下一格真实提示词可继续覆盖 prefill/decode。

In [ ]:
first = session.accelerator.generate_tokens([128000], 2)
second = session.accelerator.generate_tokens([128000], 2)
report["bos"] = {"input_tokens": [128000], "first": first, "second": second,
                 "repeat_equal": first == second}
print(report["bos"])
assert first == second, "两次输出不一致，请保留输出和完整日志"

## 6. 文本生成

可修改 prompt 并重复运行本格，复用已加载的模型。上下文上限 8192，输出遇 EOS 可提前结束。

In [ ]:
result = session.generate("Explain what an FPGA is.", max_new_tokens=16)
report["inference"] = result.to_dict()
print(result.text)
print(json.dumps(report["inference"], ensure_ascii=False, indent=2))

## 7. 保存结果

保存后，将该 JSON 与出现异常时的完整单元输出一并提供。中间步骤失败时，也可以执行本格保存目前已有的 report。

In [ ]:
log_dir = ROOT / "logs"
log_dir.mkdir(exist_ok=True)
output = log_dir / datetime.now().strftime("jupyter_board_%Y%m%d_%H%M%S.json")
output.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print("Saved:", output)